<a href="https://colab.research.google.com/github/z0n6/universal-ai-transcriber/blob/main/transcriber.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎙️ Universal AI Transcriber (萬用音訊/影片轉錄神器)

這是一個基於 `faster-whisper` 開發的高效能語音轉錄工具。利用免費的 Google Colab GPU，一鍵將 Podcast、會議記錄或影片音訊轉換為高精準度的**逐字稿**與**專業字幕檔**。

### ✨ 特色亮點
* ⚡ **極速匯出**：轉錄與檔案生成分離，可秒速匯出不同格式，無需重複等待。
* 🎯 **繁中優化**：預設使用對繁簡轉換與台灣口音辨識度最佳的 `large-v2` 模型。
* 🛠️ **全格式支援**：支援易讀文件 (Docx/TXT) 與專業字幕 (SRT/VTT)。

> 💡 **使用說明**：請依序點擊下方三個步驟的「Play 按鈕 (▶️)」即可完成轉錄。

In [ ]:
# @title 🛠️ 第一步：初始化環境 (點擊左側 Play 按鈕)
# @markdown 系統將自動安裝 Whisper / Qwen3-ASR 模型、Docx 處理工具與音訊處理庫。
# @markdown <br><font size="2" color="gray">⏳ 初次執行約需 1-2 分鐘，請耐心等候...</font>

import os
import sys
import time
from datetime import datetime
from google.colab import files
from tqdm.notebook import tqdm

# 隱藏安裝過程的輸出，保持介面乾淨
from IPython.utils import io
import warnings
warnings.filterwarnings('ignore')

print("🔄 正在安裝必要套件...", end="")

with io.capture_output() as captured:
    !pip install -q faster-whisper qwen-asr opencc-python-reimplemented python-docx yt-dlp

# 預先載入必要的 Library，避免在主程式才報錯
from faster_whisper import WhisperModel
from docx import Document
from qwen_asr import Qwen3ASRModel
from opencc import OpenCC
import torch

print()
print("✅ 安裝完成！請繼續執行下一步。")


In [ ]:
# @title 🎙️ 第二步：開始轉錄 (等待時間較長，但同一音檔僅需執行一次)

import json
import math
import os
import re
import subprocess
import inspect
from pathlib import Path
import torch

# @markdown #### **1. 輸入設定**
podcast_url = "https://rss.soundon.fm/rssf/954689a5-3096-43a4-a80b-7810b219cef3/feedurl/5fbff599-9897-422a-812f-479eb0d75bd1/rssFileVip.mp3?timestamp=1769238608784" # @param {type:"string"}

# @markdown #### **2. 模型與轉錄設定**
model_provider = "faster-whisper" # @param ["faster-whisper", "qwen3-asr"]

# @markdown - Whisper 設定
model_size = "large-v2" # @param ["medium", "large-v2", "large-v3"]
whisper_language = "zh" # @param {type:"string"}
# @markdown <font size="2" color="#0066cc"><b>💡 開發者建議：</b>繁體中文轉錄強烈建議維持 <b>large-v2</b>。經實測，v2 在台灣口音與繁簡轉換的穩定性上顯著優於 v3。</font>
initial_prompt = "繁體中文。以下是專有名詞：癌大, 孟恭, 諾亞, 主委。話題包含美股、台股與投資心法。開場白：歡迎收聽股癌，我是謝孟恭。" # @param {type:"string"}

# @markdown - Qwen3-ASR 設定
qwen_model_id = "Qwen/Qwen3-ASR-0.6B" # @param ["Qwen/Qwen3-ASR-0.6B", "Qwen/Qwen3-ASR-1.7B"]
qwen_language = "zh" # @param {type:"string"}
qwen_use_timestamps = True # @param {type:"boolean"}
qwen_max_new_tokens = 256 # @param {type:"integer"}
# @markdown <font size="2" color="gray">可輸入「背景/專有名詞」作為偏好上下文；若模型不支援將自動忽略。</font>
qwen_context = "" # @param {type:"string"}
# @markdown <font size="2" color="gray">Qwen3-ASR 對長音檔較易中斷，建議開啟切段。</font>
qwen_chunking = True # @param {type:"boolean"}
qwen_chunk_minutes = 10 # @param {type:"integer"}
qwen_min_last_chunk_minutes = 5 # @param {type:"integer"}

# @markdown - 輸出語系 (繁體/臺灣用字)
output_script = "zh-TW" # @param ["keep", "zh-Hant", "zh-TW"]

# @markdown <font size="2" color="gray">若 qwen_use_timestamps 為 False，將無法輸出 SRT/VTT（仍可輸出 TXT/DOCX）。</font>

def download_audio(url):
    print(f"⬇️ 正在下載音檔：{url} ...")
    output_filename = "podcast_audio.mp3"
    !wget -q -O {output_filename} {url}
    return output_filename

def _build_cc_converter(script):
    if not script or script == "keep":
        return None
    if script == "zh-Hant":
        return OpenCC("s2t")
    if script == "zh-TW":
        return OpenCC("s2twp")
    return None

def _run(cmd):
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

def _probe_duration(path):
    cmd = [
        "ffprobe",
        "-v",
        "error",
        "-show_entries",
        "format=duration",
        "-of",
        "default=noprint_wrappers=1:nokey=1",
        path,
    ]
    try:
        result = subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
        return float(result.stdout.strip())
    except Exception:
        return None

def _adjust_chunk_seconds(total_seconds, chunk_seconds, min_last_seconds):
    if not total_seconds or chunk_seconds <= 0:
        return chunk_seconds
    if total_seconds <= chunk_seconds:
        return chunk_seconds
    if min_last_seconds <= 0:
        return chunk_seconds

    num_chunks = math.ceil(total_seconds / chunk_seconds)
    remainder = total_seconds - (num_chunks - 1) * chunk_seconds
    if remainder >= min_last_seconds or num_chunks <= 1:
        return chunk_seconds

    num_chunks -= 1
    return int(math.ceil(total_seconds / num_chunks))

def _build_offsets(files, default_chunk_seconds):
    offsets = [0]
    running = 0.0
    for prev in files[:-1]:
        dur = _probe_duration(prev) or float(default_chunk_seconds)
        running += dur
        offsets.append(running)
    return offsets

def _split_audio_ffmpeg(path, chunk_seconds, min_last_seconds):
    if chunk_seconds <= 0:
        return [path], [0]

    total = _probe_duration(path)
    chunk_seconds = _adjust_chunk_seconds(total, chunk_seconds, min_last_seconds)

    chunk_dir = Path("qwen_chunks")
    chunk_dir.mkdir(exist_ok=True)
    for old in chunk_dir.glob("chunk_*.wav"):
        old.unlink()

    pattern = str(chunk_dir / "chunk_%03d.wav")
    cmd = [
        "ffmpeg",
        "-y",
        "-i",
        path,
        "-ar",
        "16000",
        "-ac",
        "1",
        "-f",
        "segment",
        "-segment_time",
        str(chunk_seconds),
        "-reset_timestamps",
        "1",
        pattern,
    ]
    try:
        _run(cmd)
    except Exception:
        print("⚠️ 找不到 ffmpeg 或切段失敗，改用整段轉錄。")
        return [path], [0]

    files = sorted(chunk_dir.glob("chunk_*.wav"))
    if not files:
        return [path], [0]

    offsets = _build_offsets([str(f) for f in files], chunk_seconds)
    return [str(f) for f in files], offsets

def _extract_ts_item(item):
    if isinstance(item, dict):
        start = item.get("start") or item.get("start_time") or item.get("start_time_sec")
        end = item.get("end") or item.get("end_time") or item.get("end_time_sec")
        text = item.get("text") or item.get("token") or item.get("word") or item.get("char")
        return start, end, text
    if isinstance(item, (list, tuple)) and len(item) >= 3:
        if isinstance(item[0], (int, float)) and isinstance(item[1], (int, float)):
            return item[0], item[1], item[2]
        if isinstance(item[1], (int, float)) and isinstance(item[2], (int, float)):
            return item[1], item[2], item[0]
    return None, None, None

def _build_segments_from_time_stamps(time_stamps):
    if not time_stamps:
        return []

    segments = []
    buffer = []
    seg_start = None
    seg_end = None
    punct = re.compile(r"[。！？!?…]$")

    for item in time_stamps:
        start, end, text = _extract_ts_item(item)
        if start is None or end is None or text is None:
            continue
        if seg_start is None:
            seg_start = float(start)
        seg_end = float(end)
        buffer.append(str(text))
        joined = "".join(buffer).strip()
        if punct.search(str(text)) or (seg_end - seg_start) >= 12:
            if joined:
                segments.append({"start": seg_start, "end": seg_end, "text": joined})
            buffer = []
            seg_start = None
            seg_end = None

    if buffer and seg_start is not None and seg_end is not None:
        segments.append({"start": seg_start, "end": seg_end, "text": "".join(buffer).strip()})

    return segments

def _call_qwen_transcribe(model, audio_path, language, return_time_stamps, context_text):
    kwargs = dict(
        audio=audio_path,
        language=language,
        return_time_stamps=return_time_stamps,
    )
    if context_text:
        try:
            sig = inspect.signature(model.transcribe)
            params = sig.parameters
            if "context" in params:
                kwargs["context"] = context_text
            elif "prompt" in params:
                kwargs["prompt"] = context_text
            elif "hotwords" in params:
                kwargs["hotwords"] = context_text
        except Exception:
            pass
    return model.transcribe(**kwargs)

# 主執行邏輯
if podcast_url:
    try:
        audio_file = download_audio(podcast_url)

        if model_provider == "faster-whisper":
            print(f"🚀 載入模型 ({model_size})... 請稍候")
            model = WhisperModel(model_size, device="cuda", compute_type="float16")

            print("✍️ 開始轉錄... (請耐心等候，完成後資料將暫存)")
            segments, info = model.transcribe(
                audio_file,
                beam_size=5,
                initial_prompt=initial_prompt,
                language=(whisper_language or None),
            )

            total_duration = info.duration
            pbar = tqdm(total=round(total_duration), unit="sec", desc="轉錄進度")

            # 將轉錄結果暫存為標準化字典列表
            raw_results = []
            for segment in segments:
                raw_results.append({
                    "start": segment.start,
                    "end": segment.end,
                    "text": segment.text.strip()
                })
                pbar.n = min(round(segment.end), round(total_duration))
                pbar.refresh()
            pbar.close()
            has_timestamps = True
            meta = {
                "provider": model_provider,
                "model": model_size,
                "language": whisper_language,
                "has_timestamps": has_timestamps,
            }
        else:
            print(f"🚀 載入 Qwen3-ASR 模型 ({qwen_model_id})... 請稍候")
            qwen_kwargs = dict(
                dtype=torch.float16,
                device_map="cuda:0",
                max_inference_batch_size=8,
                max_new_tokens=qwen_max_new_tokens,
            )
            if qwen_use_timestamps:
                qwen_kwargs["forced_aligner"] = "Qwen/Qwen3-ForcedAligner-0.6B"
                qwen_kwargs["forced_aligner_kwargs"] = dict(
                    dtype=torch.float16,
                    device_map="cuda:0",
                )

            model = Qwen3ASRModel.from_pretrained(qwen_model_id, **qwen_kwargs)

            print("✍️ 開始轉錄... (請耐心等候，完成後資料將暫存)")
            chunk_seconds = max(1, int(qwen_chunk_minutes) * 60) if qwen_chunking else 0
            min_last_seconds = max(0, int(qwen_min_last_chunk_minutes) * 60) if qwen_chunking else 0
            chunk_files, offsets = _split_audio_ffmpeg(audio_file, chunk_seconds, min_last_seconds)

            raw_results = []
            has_timestamps = qwen_use_timestamps
            missing_timestamps = False

            for chunk_path, offset in zip(chunk_files, offsets):
                results = _call_qwen_transcribe(
                    model,
                    audio_path=chunk_path,
                    language=(qwen_language or None),
                    return_time_stamps=qwen_use_timestamps,
                    context_text=qwen_context.strip(),
                )

                primary = results[0] if results else None
                if primary and qwen_use_timestamps:
                    segments = _build_segments_from_time_stamps(getattr(primary, "time_stamps", None))
                else:
                    segments = []

                if segments:
                    for seg in segments:
                        seg["start"] += offset
                        seg["end"] += offset
                    raw_results.extend(segments)
                else:
                    text = getattr(primary, "text", "") if primary else ""
                    text = text.strip()
                    if text:
                        raw_results.append({"start": None, "end": None, "text": text})
                    missing_timestamps = True

            if missing_timestamps:
                has_timestamps = False

            if not raw_results:
                raw_results = [{"start": None, "end": None, "text": ""}]
                has_timestamps = False

            meta = {
                "provider": model_provider,
                "model": qwen_model_id,
                "language": qwen_language,
                "has_timestamps": has_timestamps,
                "chunking": bool(qwen_chunking),
                "chunk_minutes": int(qwen_chunk_minutes) if qwen_chunking else None,
                "min_last_chunk_minutes": int(qwen_min_last_chunk_minutes) if qwen_chunking else None,
            }
            if qwen_context.strip():
                meta["context"] = "provided"

        # 轉為繁體 / 台灣用字
        cc = _build_cc_converter(output_script)
        if cc:
            for item in raw_results:
                item["text"] = cc.convert(item["text"])
            meta["output_script"] = output_script

        # 儲存為 JSON，供下一步使用
        with open("transcription_raw.json", "w", encoding="utf-8") as f:
            json.dump(raw_results, f, ensure_ascii=False, indent=2)

        with open("transcription_meta.json", "w", encoding="utf-8") as f:
            json.dump(meta, f, ensure_ascii=False, indent=2)

        print("
✅ 轉錄核心工作完成！原始資料已暫存。請至「第三步」選擇下載格式。")

    except Exception as e:
        print(f"❌ 發生錯誤: {e}")
else:
    print("⚠️ 請輸入 Podcast 網址。")


In [ ]:
# @title 📥 第三步：自訂檔名與下載 (秒速完成)
import json
import os
import math
import re
from datetime import datetime
from google.colab import files
from docx import Document

# @markdown #### **1. 設定輸出檔名**
# @markdown <font size="2" color="gray">請輸入檔案名稱 (系統會自動補上副檔名)</font>
output_filename_base = "EP630" # @param {type:"string"}

# @markdown ---
# @markdown #### **2. 選擇需要的檔案格式 (可複選)**
export_srt = False # @param {type:"boolean"}
export_vtt = False # @param {type:"boolean"}
export_txt = False # @param {type:"boolean"}
export_docx = True # @param {type:"boolean"}

def format_timestamp(seconds, format_type="standard"):
    """轉換秒數為不同格式的時間戳"""
    if seconds is None:
        return None
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    secs = int(seconds % 60)

    if format_type == "srt":
        millis = int((seconds - int(seconds)) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d},{millis:03d}"
    elif format_type == "vtt":
        millis = int((seconds - int(seconds)) * 1000)
        return f"{hours:02d}:{minutes:02d}:{secs:02d}.{millis:03d}"
    else: # docx, txt 使用的易讀格式
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

def sanitize_filename(filename):
    """防呆機制：清理檔名中的不合法字元，若為空則用時間戳代入"""
    if not filename.strip():
        return f"transcript_{datetime.now().strftime('%Y%m%d_%H%M')}"
    # 移除 Windows/Linux 檔名不允許的字元 \ / : * ? " < > |
    return re.sub(r'[\/:*?"<>|]', '_', filename)


# 檢查是否有轉錄資料
if not os.path.exists("transcription_raw.json"):
    print("⚠️ 找不到轉錄資料，請先執行「第二步」！")
else:
    # 讀取轉錄資料
    with open("transcription_raw.json", "r", encoding="utf-8") as f:
        segments = json.load(f)

    has_timestamps = None
    if os.path.exists("transcription_meta.json"):
        with open("transcription_meta.json", "r", encoding="utf-8") as f:
            meta = json.load(f)
        has_timestamps = meta.get("has_timestamps")

    if has_timestamps is None:
        has_timestamps = all(
            seg.get("start") is not None and seg.get("end") is not None
            for seg in segments
        )

    download_list = []

    # 處理檔名
    safe_base_name = sanitize_filename(output_filename_base)
    print(f"📂 準備產生檔案，基礎檔名為: {safe_base_name}")

    if (export_srt or export_vtt) and not has_timestamps:
        print("⚠️ 目前轉錄結果沒有時間戳，無法輸出 SRT/VTT。請在第二步啟用 qwen_use_timestamps 或改用 Whisper。")
        export_srt = False
        export_vtt = False

    # 1. 產生 SRT (標準字幕)
    if export_srt:
        srt_name = f"{safe_base_name}.srt"
        with open(srt_name, "w", encoding="utf-8") as f:
            for i, seg in enumerate(segments, start=1):
                start = format_timestamp(seg['start'], "srt")
                end = format_timestamp(seg['end'], "srt")
                if start is None or end is None:
                    continue
                f.write(f"{i}
{start} --> {end}
{seg['text']}

")
        download_list.append(srt_name)

    # 2. 產生 VTT (網頁影片常用字幕)
    if export_vtt:
        vtt_name = f"{safe_base_name}.vtt"
        with open(vtt_name, "w", encoding="utf-8") as f:
            f.write("WEBVTT

")
            for seg in segments:
                start = format_timestamp(seg['start'], "vtt")
                end = format_timestamp(seg['end'], "vtt")
                if start is None or end is None:
                    continue
                f.write(f"{start} --> {end}
{seg['text']}

")
        download_list.append(vtt_name)

    # 3. 產生 DOCX (易讀文件)
    if export_docx:
        doc_name = f"{safe_base_name}.docx"
        doc = Document()
        doc.add_heading(f'轉錄內容 - {safe_base_name}', 0)
        for seg in segments:
            p = doc.add_paragraph()
            if has_timestamps:
                start = format_timestamp(seg['start'], "standard")
                p.add_run(f"[{start}] ").bold = True
            p.add_run(seg['text'])
        doc.save(doc_name)
        download_list.append(doc_name)

    # 4. 產生 TXT (純文字備份)
    if export_txt:
        txt_name = f"{safe_base_name}.txt"
        with open(txt_name, "w", encoding="utf-8") as f:
            for seg in segments:
                if has_timestamps:
                    start = format_timestamp(seg['start'], "standard")
                    f.write(f"[{start}] {seg['text']}
")
                else:
                    f.write(f"{seg['text']}
")
        download_list.append(txt_name)

    # 觸發下載
    if download_list:
        print(f"✅ 完成！共產生 {len(download_list)} 個檔案，即將下載。")
        for file in download_list:
            files.download(file)
    else:
        print("⚠️ 未勾選任何輸出格式。")
